In [165]:
import numpy as np
rng = np.random.default_rng()

n = 10
m = 5
o = 7

w_strength = 0.4
rho_strength = 1
alpha_strength = 1.4
tau_strength = 0

w = np.ones((n, m, m))*w_strength
rho = np.ones((n, m, n))*rho_strength
alpha = np.ones((n, m, n))*alpha_strength
tau = np.ones((n,m))*tau_strength

v = np.linspace(-1,1,o) # Candidate states
b = np.random.choice(v, size=(n,m)) # belief spins
s = np.random.choice(v, (n,m,n)) # social spins


beta_pers = 1
beta_soc = 1
beta_ext = 1

In [168]:
# H_pers
F = tau + np.einsum('ijk,ij->ik', w, b) #Local pers field
H_b = - F*b
H_pers = - F[:, :, None] * v[None, None, :]
delta_H_pers = H_b[:, :, None] - H_pers


In [ ]:
# H_soc 
F = np.einsum('jik,jik->ji', rho, s)        # (J, I): social field on each personal belief
H_b = - F*b
H_soc = -F[:, :, None]* v[None, None, :]     # (J, I, O): energy of every candidate state
delta_H_soc = H_b[:, :, None] - H_soc

In [171]:
# delta D_pers
delta_D_pers = beta_pers*delta_H_pers + beta_soc*delta_H_soc

delta_D_pers.shape

(10, 5, 7)

In [206]:
logits = 1 / (1 + np.exp(delta_D_pers))
normalize = logits.sum(axis=-1)
glauber_prob = logits / normalize[:, :, None]
glauber_prob.shape

(10, 5, 7)

In [205]:
# Sample
u = np.random.uniform(size = (n, m, 1))
cumsum = glauber_prob.cumsum(axis=2)
indx = (cumsum < u).sum(axis=2)
new_b = v[indx]

new_b


array([[ 0.66666667,  1.        ,  0.        ,  1.        ,  0.        ],
       [ 0.66666667,  1.        , -0.33333333,  0.        , -0.33333333],
       [-0.33333333,  0.66666667,  1.        ,  1.        ,  1.        ],
       [-1.        ,  1.        ,  0.33333333, -1.        , -1.        ],
       [ 1.        ,  1.        ,  1.        , -1.        , -0.33333333],
       [ 1.        ,  0.66666667,  0.66666667, -1.        ,  0.66666667],
       [ 0.33333333, -1.        ,  1.        ,  0.33333333, -0.33333333],
       [-0.66666667,  0.66666667, -0.66666667,  0.        ,  1.        ],
       [ 1.        , -0.66666667, -0.66666667, -0.66666667,  0.66666667],
       [ 0.66666667, -1.        , -1.        ,  0.        ,  0.        ]])

In [214]:
rho.shape

(10, 5, 10)

In [222]:
F_soc = - rho*b[:, :, None]
H_s = F_soc*s
H_soc_candidates = F_soc[:, :, :, None]*v[None, None, None, :]
delta_H_soc = H_s[:, :, :, None] - H_soc_candidates
delta_H_soc.shape


(10, 5, 10, 7)

In [219]:
H_candidates.shape

(10, 5, 10, 7)

In [229]:
H_ext_candidates.shape

(10, 5, 10, 7)